# 🚀 Build a shopping agent with Sequence-Level Knowledge Distillation on Amazon SageMaker AI

This notebook teaches a small, inexpensive model to act as an agentic **shopping assistant** by learning from the decoded trajectories of a stronger "teacher" agent — a technique called **Sequence-Level Knowledge Distillation (SeqKD)**.

### Why distillation?
A large frontier model can solve agentic tool-use tasks well, but is slow and expensive to run in production. SeqKD lets you transfer that capability into a **4B-parameter student** (`Qwen/Qwen3.5-4B`) that is cheap to host and fast to serve, while retaining most of the teacher's task skill. Mechanically, SeqKD is simply **supervised fine-tuning (SFT)** on the teacher's successful trajectories — so we run it with SageMaker's **serverless `SFTTrainer`** (SageMaker Python SDK v3), which handles the training infrastructure for you.

### The task
A shopper asks for products in natural language — e.g. *"women's jeans in size eu:30 with a high waist, priced above 114 PHP"*. The agent must use four tools to search a **2.7-million-product catalog**, inspect candidates, verify every constraint, and recommend matching products:

| Tool | Purpose |
|---|---|
| `find_product` | Search the catalog |
| `view_product_information` | Inspect a candidate's details |
| `recommend_product` | Return the final recommendation |
| `terminate` | End the episode |

The catalog, test sets, and the success metric come from **[ShoppingBench](https://github.com/yjwjy/ShoppingBench)** ([paper](https://arxiv.org/abs/2508.04266)). The teacher trajectories come from **[oro-ai/sn15-shoppingbench-sft-15k](https://huggingface.co/datasets/oro-ai/sn15-shoppingbench-sft-15k)**.

### What you will do
1. **Set up** your environment and SageMaker session.
2. **Prepare the data** — encode each teacher trajectory step into the exact prompt/response format the ShoppingBench agent loop uses, then register it as a versioned SageMaker dataset.
3. **Train** the student with the serverless `SFTTrainer` (LoRA).
4. **Retrieve** the registered, deploy-ready Model Package.
5. **Evaluate** the student in the two companion notebooks: `2-tool-call-accuracy-gate.ipynb` (a cheap, fast per-step tool-call accuracy gate — run it first) and `3-evaluate-shopping-agent.ipynb` (the full ShoppingBench ASR and IFEval run).

> **Reference result (held-out, thinking off):** the student's mean Agent Success Rate rises from **0.085 (base) → 0.346 (student)** across the three ShoppingBench intents — a large gain in tool-use capability while general instruction-following is preserved. (Numbers are from one sample run and will vary with training data, randomness, and model/endpoint versions.)

## 00. Setup

Install the pinned dependencies and configure the SageMaker session, execution role, and S3 location used throughout the notebook. Run the install cell **first**, then restart the kernel so the pinned package versions are the ones imported.

In [ ]:
# Install dependencies, pinned to the versions resolved at authoring time (reproducible).
# sagemaker: v3 SDK (latest at authoring). huggingface_hub/transformers: teacher dataset download and the student
# tokenizer (step 01). The rest are ShoppingBench harness runtime deps (step 01b).
%pip install -q \
    "sagemaker==3.23.0" \
    "huggingface_hub==1.33.0" \
    "transformers==5.19.0" \
    "ujson==5.13.0" \
    "pydantic==2.13.5" \
    "portalocker==4.4.0" \
    "openai==1.109.1" \
    "duckduckgo-search==8.1.1" \
    "googlesearch-python==1.3.0" \
    "mcp==1.28.1" \
    "colorama==0.4.6"
# NOTE: restart the kernel after this install so the pinned versions are imported.

**Configure the session.** This cell resolves everything downstream steps depend on:

- **Region** — taken from your environment (defaults to `us-west-2`).
- **Execution role** — the IAM role SageMaker assumes to run training and register artifacts. If auto-detection fails (e.g. running locally), replace the placeholder ARN with your own.
- **S3 location** — training data and model outputs are staged under `s3://<default-bucket>/seqkd-shopping/`. We use the SageMaker session's default bucket; replace it if you prefer a specific bucket.
- **MLflow endpoint** — set so training/validation metrics stream to the managed MLflow app while the job runs.

In [ ]:
import os, json, boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

REGION = boto3.Session().region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION

# SageMaker session (SDK v3)
sagemaker_session = Session(boto_session=boto3.Session(region_name=REGION))

try:
    ROLE = get_execution_role()
    print("Execution role:", ROLE)
except Exception:
    ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
    ROLE = f"arn:aws:iam::{ACCOUNT}:role/<your-sagemaker-execution-role>"   #replace with your unique execution role

# Use the SageMaker session's default bucket; fall back to a placeholder if unavailable
try:
    BUCKET = sagemaker_session.default_bucket()
except Exception:
    BUCKET = "<your-bucket>" #replace with your bucket
    
PREFIX  = "seqkd-shopping"
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
print(REGION, ROLE, S3_ROOT)

## 01. Data preparation

The student learns by imitating the teacher's successful tool-use trajectories. In this step we (a) download those trajectories, (b) fetch the ShoppingBench harness so we can format examples exactly as its agent loop expects, (c) encode each step into a `{prompt, completion}` training example, and (d) register the result as versioned SageMaker datasets.

### 01a. Load the teacher trajectories and the ShoppingBench harness

The teacher trajectories come from **[oro-ai/sn15-shoppingbench-sft-15k](https://huggingface.co/datasets/oro-ai/sn15-shoppingbench-sft-15k)**. They were produced in an SN15 subnet arena where competing agents submit candidate trajectories and a reasoning judge selects the winner; the winners are then filtered for structural quality to form the teacher corpus.

The cell below also runs `scripts/setup_shopbench.sh --code-only`, which checks out the **ShoppingBench** repo at a **pinned commit** — so the harness code we execute and import is a fixed, reviewable snapshot (reproducibility + integrity). We reuse the harness's own prompt builder in 01b to guarantee our training examples match inference byte-for-byte. The printout reports how many test problems are genuinely *held out* (their query never appears in the teacher corpus) — important for a fair evaluation later.

One implementation detail worth knowing: the harness source is *appended* to `sys.path`, not prepended, because it imports its siblings unqualified (`from util.message import …`) and so `src/agent` must be importable — but appending ensures this downloaded third-party code can never shadow the standard library or installed packages.

In [ ]:
import json, sys, subprocess
from pathlib import Path
from huggingface_hub import list_repo_files, hf_hub_download

HF_STUDENT = "Qwen/Qwen3.5-4B"
REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # teacher trajectories

# ---- Load the teacher trajectories ----
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")

# ---- Fetch the ShoppingBench harness (pinned commit, code only — no catalog/index) ----
# Paths are relative to this notebook's folder (the kernel runs here).
SHOPBENCH_WORK = Path("shopbench-eval")
SHOPBENCH_WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(["bash", "scripts/setup_shopbench.sh", "--code-only", str(SHOPBENCH_WORK)], check=True)
SB_DIR = SHOPBENCH_WORK / "ShoppingBench"

# The harness imports its siblings unqualified (`from util.message import ...`), so src/agent must be on
# sys.path. APPEND so this third-party code can't shadow stdlib/installed packages.
AGENT_SRC = str((SB_DIR / "src" / "agent").resolve())
if AGENT_SRC not in sys.path:
    sys.path.append(AGENT_SRC)

# ---- Report how many test problems are genuinely held out (query not in the teacher corpus) ----
test_rows = {t: [json.loads(l) for l in open(SB_DIR / "data" / f"synthesize_{t}_test.jsonl") if l.strip()]
             for t in ("product", "shop", "voucher")}

def traj_query(t):                           # a trajectory's task == its first user message
    return next((m["content"] for m in t["messages"] if m.get("role") == "user"), None)

train_queries = {traj_query(t) for t in raw}
for t, rows in test_rows.items():
    n = sum(r["query"] not in train_queries for r in rows)
    print(f"{t}: {n} / {len(rows)} test problems are held out (query not in the corpus)")

### 01b. Encode every trajectory step in the ShoppingBench harness format

A trajectory is a multi-step episode (search → inspect → verify → recommend). We turn **each assistant action into one training example**, formatted exactly as the ShoppingBench agent loop will present it at inference time. Matching that format precisely is critical: if training prompts differ from what the harness sends during evaluation, the student's learned behavior won't transfer.

Each example is a single prompt/response pair:

| Message | Content |
|---|---|
| `user` (prompt) | The agent instructions + tool list, followed by the dialogue history so far (the original request plus earlier tool calls and their results) |
| `assistant` (target) | The teacher's next action: `<tool_call>[{"name": …, "parameters": …}]</tool_call>` |

Key choices made in the cell:

- **Loss on the completion only** — `SFTTrainer` masks the prompt, so the model is trained to produce the *action*, not to reproduce the instructions.
- **Argument normalization** — teacher tool arguments are reshaped to exactly what the harness tools accept (e.g. `product_ids` as a comma-separated string, `find_product` requires a `page`).
- **Drop out-of-scope trajectories** — any trajectory using a tool the harness doesn't have is skipped.
- **Split by query, not by step** — ~30 trajectories can share the same shopper query, so splitting at the step level would leak near-duplicates across train/val. We hold out ~5% of *queries* instead.
- **Self-check** — we assert the harness's own parser accepts essentially 100% of the generated targets, catching format drift before training.

In [ ]:
# The harness source path (SB_DIR/src/agent) was added to sys.path in the checkout cell above.
import random
from run_rollout import get_system_prompt, get_user_prompt
from util.message import Message, generate_tool_call_id
from transformers import AutoTokenizer

SYSTEM_PROMPT = get_system_prompt({"system_prompt_file": str(SB_DIR / "src/agent/prompt/rollout.md"),
                                   "exclude_tools": ["web_search"]})   # web search: knowledge intent only
HARNESS_TOOLS = {"find_product", "view_product_information", "recommend_product", "terminate"}
tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)

def harness_call(tc):
    """ORO tool call -> the argument shapes the harness tools accept."""
    f = tc.get("function", tc); name = f["name"]; a = f.get("arguments") or {}
    a = json.loads(a) if isinstance(a, str) else dict(a)
    if name == "find_product":
        if "q" not in a and "query" in a: a["q"] = a.pop("query")
        a.setdefault("page", 1)                                   # required by the harness's find_product
    elif name in ("view_product_information", "recommend_product"):
        v = a.pop("product_ids", a.pop("product_id", ""))
        a["product_ids"] = ",".join(map(str, v)) if isinstance(v, list) else str(v)   # comma-separated string
    elif name == "terminate":
        s = a.get("status", "success"); a["status"] = s if isinstance(s, str) else ("success" if s else "failure")
    return {"name": name, "parameters": a}

def to_rows(traj):
    msgs = [m for m in traj["messages"] if m.get("role") != "system"]
    query = next(m["content"] for m in msgs if m["role"] == "user")
    steps, i = [], 0
    while i < len(msgs):                     # each tool-calling assistant turn + the tool results after it
        m = msgs[i]; i += 1
        if m["role"] != "assistant" or not m.get("tool_calls"):
            continue                         # prose-only turns never occur in the harness loop
        calls = [harness_call(tc) for tc in m["tool_calls"]]
        if any(c["name"] not in HARNESS_TOOLS for c in calls):
            return None                      # uses a tool the ShoppingBench harness does not have
        results = []
        while i < len(msgs) and msgs[i]["role"] == "tool":
            r = msgs[i]["content"]
            try: r = json.loads(r)
            except Exception: pass
            results.append(r); i += 1
        steps.append((calls, results))
    if not steps:
        return None
    rows, history, prev = [], [], Message(user=query)
    for calls, results in steps:
        user_prompt = get_user_prompt(prev, history)               # appends prev to history, as the agent loop does
        # instructions at the top of the user message (see 01b above)
        rows.append({"messages": [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_prompt}"},
                                  {"role": "assistant", "content": f"<tool_call>{json.dumps(calls, ensure_ascii=False)}</tool_call>"}]})
        tcs = [dict(c, tool_call_id=generate_tool_call_id(c["name"], c["parameters"])) for c in calls]
        prev = Message(tool_call=tcs, obs=[{"tool_call_id": t["tool_call_id"], "results": r} for t, r in zip(tcs, results)])
        if any(c["name"] == "terminate" for c in calls):
            break                                                   # the harness loop stops here
    return rows

# Split by query (~30 trajectories share each query, so a step-level split would leak)
by_q = {}
for t in raw: by_q.setdefault(traj_query(t), []).append(t)
queries = sorted(by_q); random.seed(42); random.shuffle(queries)
val_q = set(queries[:max(1, len(queries) // 20)])                  # ~5% of queries held out
train, val, skipped = [], [], 0
for q, trajs in by_q.items():
    for t in trajs:
        rows = to_rows(t)
        if rows is None: skipped += 1; continue
        (val if q in val_q else train).extend(rows)
random.shuffle(train)
print(f"{len(raw)} trajectories ({skipped} skipped: non-harness tools or no tool calls) -> "
      f"train {len(train)} / val {len(val)} examples; {len(queries) - len(val_q)} / {len(val_q)} queries")

# Self-check: the harness parser must accept every target
target = lambda row: row["messages"][-1]["content"]
rate = sum(bool(Message.from_string("", target(r)).tool_call) for r in train[:2000]) / min(len(train), 2000)
print(f"harness parser accepts {rate:.1%} of targets"); assert rate > 0.99

### Filter by length, then write the dataset files

The trainer enforces a maximum sequence length (`dataset_max_len`, set in step 02) and **silently left-truncates** anything longer — which would chop the instructions off the front of an example and corrupt the target. To avoid that, we measure each example's token count and **drop** any that exceed the limit, rather than letting them be truncated. The cell also prints the length distribution (median / p99 / max) so you can sanity-check the limit, shows one fully-rendered example as the trainer will see it, and writes the train/val splits to local `.jsonl` files for upload.

In [ ]:
# Drop examples longer than dataset_max_len: the trainer truncates from the left
MAX_TOKENS = 16384  # = dataset_max_len in step 02

def token_lengths(rows, chunk=1000):
    # Render once, then tokenize in batched chunks (fast-tokenizer parallelism) to bound peak memory.
    lens = []
    for i in range(0, len(rows), chunk):
        texts = [tok.apply_chat_template(r["messages"], tokenize=False) for r in rows[i:i + chunk]]
        lens.extend(len(ids) for ids in tok(texts)["input_ids"])
    return lens

for name, rows in (("train", train), ("val", val)):
    lens = token_lengths(rows)
    keep = [r for r, n in zip(rows, lens) if n <= MAX_TOKENS]
    s = sorted(lens)
    print(f"{name}: tokens per example median {s[len(s) // 2]}, p99 {s[int(len(s) * 0.99)]}, max {s[-1]}; "
          f"\n dropped {len(rows) - len(keep)} longer than {MAX_TOKENS}")
    rows[:] = keep

print(f"--- example (as the trainer renders it) ---\n {tok.apply_chat_template(train[0]['messages'], tokenize=False)}")

# Write the files to local disk under data/ (git-ignored).
from pathlib import Path
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
TRAIN_FILE, VAL_FILE = str(DATA_DIR / "seqkd_train.jsonl"), str(DATA_DIR / "seqkd_val.jsonl")

def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(train, TRAIN_FILE)
write_jsonl(val, VAL_FILE)

### Upload and register the datasets

Upload the two `.jsonl` files to S3, then register each with `DataSet.create`. Registration produces a **versioned dataset ARN** in the SageMaker AI Registry rather than a bare S3 path. The serverless `SFTTrainer` consumes these ARNs directly, which gives you lineage and reproducibility — every training job records exactly which dataset version it trained on.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from pathlib import Path

s3 = boto3.client("s3", region_name=REGION)
# TRAIN_FILE/VAL_FILE are local paths under data/; use just the basename for the S3 key so the
# object lands at {PREFIX}/data/<name>.jsonl (not a doubled data/data/).
TRAIN_KEY = f"{PREFIX}/data/{Path(TRAIN_FILE).name}"
VAL_KEY   = f"{PREFIX}/data/{Path(VAL_FILE).name}"
s3.upload_file(TRAIN_FILE, BUCKET, TRAIN_KEY)
s3.upload_file(VAL_FILE,   BUCKET, VAL_KEY)
TRAIN_S3 = f"s3://{BUCKET}/{TRAIN_KEY}"
VAL_S3   = f"s3://{BUCKET}/{VAL_KEY}"

train_dataset = DataSet.create(name="seqkd-shopping-train", source=TRAIN_S3)
val_dataset   = DataSet.create(name="seqkd-shopping-val",   source=VAL_S3)
print("train:", train_dataset.arn)
print("val:  ", val_dataset.arn)

## 02. Train the student with the serverless `SFTTrainer`

As noted earlier, SeqKD reduces to ordinary supervised fine-tuning on the teacher's trajectories — so training is a standard `SFTTrainer` job. The serverless trainer provisions and tears down the compute for you; you only define the model, data, and hyperparameters.

The cell below creates the trainer and a **Model Package Group** — a named container in the Model Registry that will hold the resulting model version. We train with **LoRA** (low-rank adapters), which is far cheaper than full fine-tuning and sufficient for this task; switch to `TrainingType.FULL` only if you have a reason to update all weights.

> **Licensing:** `accept_eula=True` accepts the base model's license. Review the model's terms before running. Call `SFTTrainer.list_supported_models()` to see which base models serverless training supports.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.resources import ModelPackageGroup

MPG_NAME = "seqkd-qwen35-4b-shopping"
try:
    model_package_group = ModelPackageGroup.create(model_package_group_name=MPG_NAME)
except Exception as e:
    print("reusing existing group:", type(e).__name__)
    model_package_group = MPG_NAME          # an existing name or ARN is also accepted

# SFTTrainer.list_supported_models() lists the models serverless training supports

sft_trainer = SFTTrainer(
    model="huggingface-vlm-qwen3-5-4b",     # Qwen3.5-4B
    training_type=TrainingType.LORA,        # Use TrainingType.FULL for full fine-tuning
    model_package_group=model_package_group,
    mlflow_experiment_name="seqkd-shopping-exp",
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=f"{S3_ROOT}/output/",
    accept_eula=True,                       #To accept licensing for the models
)

### Hyperparameters

Each supported model + training type ships with a tuned **recipe default**. The first cell prints those defaults; the second overrides only what we want to change for this run. Set overrides **after** creating the trainer — re-creating the trainer resets them.

The overrides used here and why:

- `dataset_max_len = 16384` — matches the length filter from step 01 so long multi-step episodes aren't truncated.
- `max_epochs = 1` — one pass over the trajectories; enough to learn the behavior without overfitting on ~38k examples.
- `global_batch_size = 64`, `learning_rate = 1e-4`, `lr_warmup_steps_ratio = 0.03` — stable settings for LoRA SFT at this scale.
- `lora_rank = 64`, `lora_alpha = 256` (α/rank = 4) — higher-capacity adapters, which this reference configuration found improves tool-use success.
- `train_val_split_ratio = 0.9` — how the trainer splits within the training channel for its internal eval.

Any unsupported value raises an error that reports the allowed range, so it's safe to experiment.

In [ ]:
from pprint import pprint
print("Default hyperparameters")
pprint(sft_trainer.hyperparameters.to_dict())

In [ ]:
sft_trainer.hyperparameters.dataset_max_len       = 16384
sft_trainer.hyperparameters.max_epochs            = 1
sft_trainer.hyperparameters.global_batch_size     = 64
sft_trainer.hyperparameters.learning_rate         = 0.0001
sft_trainer.hyperparameters.lora_rank             = 64
sft_trainer.hyperparameters.lora_alpha            = 256     # alpha/rank = 4
sft_trainer.hyperparameters.train_val_split_ratio = 0.9
sft_trainer.hyperparameters.lr_warmup_steps_ratio = 0.03
pprint(sft_trainer.hyperparameters.to_dict())

### Validate, then launch training

Always **dry-run first**: `train(dry_run=True)` validates your IAM permissions, dataset ARNs, and hyperparameters without provisioning any compute — a fast way to catch misconfiguration before paying for a job.

Once the dry run passes, the next cell launches the real job. `wait=True` blocks and streams logs until completion; use `wait=False` if you'd rather submit and poll later. Training is serverless, so there's no instance to manage — expect the job to take on the order of a couple of hours depending on data size and the chosen hyperparameters. Metrics stream to the MLflow app configured in step 00.

In [ ]:
sft_trainer.train(dry_run=True)
print("Dry run passed - configuration is valid.")

In [ ]:
training_job = sft_trainer.train(wait=True)
print(f"Job Name: {training_job.training_job_name} Job Status: {training_job.training_job_status}")

### Inspect the training loss curve

Once the job completes, plot the loss from MLflow to sanity-check the run. The MLflow app, experiment, and run are resolved **at run time** and recorded on the training job, so the cell below reads them off the job rather than hardcoding anything (for the reference run that resolves to experiment `seqkd-shopping-exp`, run `huggingface-vlm-qwen3-5-4b-sft-...`).

A healthy run shows **training loss trending down and flattening** as the model learns the teacher's behavior.

> **On validation loss:** this recipe logs the *training* loss (`train/loss`) but does not emit a separate validation-loss series to MLflow, so there's no val curve to overlay here. The authoritative generalization signal for this workflow is the **held-out ASR + IFEval evaluation in step 04** — those run on queries the model never trained on, which is a stronger check of overfitting than a val-loss curve would be.

In [ ]:
import matplotlib.pyplot as plt
from sagemaker.core.resources import TrainingJob

def plot_training_loss(trainer):
    job_name = getattr(trainer, "_latest_training_job", None) or training_job.training_job_name
    job_name = job_name if isinstance(job_name, str) else job_name.training_job_name
    job = TrainingJob.get(training_job_name=job_name)

    cfg, det = getattr(job, "mlflow_config", None), getattr(job, "mlflow_details", None)
    arn    = getattr(cfg, "mlflow_resource_arn", None)
    run_id = getattr(det, "mlflow_run_id", None)
    if not arn or not run_id:
        print("MLflow run not available yet (job still running or MLflow not configured).")
        return

    import mlflow
    from mlflow.tracking import MlflowClient
    mlflow.set_tracking_uri(arn)
    client = MlflowClient()
    print(f"experiment: {getattr(cfg, 'mlflow_experiment_name', '?')}  run: {getattr(cfg, 'mlflow_run_name', '?')}")

    loss = sorted(client.get_metric_history(run_id, "train/loss"), key=lambda h: h.step)
    if not loss:
        print(f"No 'train/loss' logged on run {run_id} yet.")
        return

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot([h.step for h in loss], [h.value for h in loss], linewidth=2, label="train/loss")
    ax.set_xlabel("Global step"); ax.set_ylabel("Loss")
    ax.set_title(f"SeqKD training loss — {job_name}", fontweight="bold")
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

try:
    plot_training_loss(sft_trainer)
except Exception as e:
    print("Could not fetch loss curve:", type(e).__name__, e)

## 03. Retrieve the trained Model Package

When the job finishes it registers a **Model Package** in the group created in step 02. For a LoRA run, SageMaker has already **merged the adapter into the base weights**, so the package is a complete, ready-to-deploy model — no separate merge step needed.

This cell fetches the Model Package ARN from the finished training job and prints its status. **Copy that ARN** — you'll paste it into the evaluation notebook (step 04). The commented AWS CLI command shows how to look up the latest package later if you no longer have the job object in memory.

In [ ]:
from sagemaker.core.resources import TrainingJob, ModelPackage

job = TrainingJob.get(training_job_name=training_job.training_job_name)
MODEL_PKG_ARN = job.output_model_package_arn
print(f"Output Model Package ARN: {MODEL_PKG_ARN}")

model_package = ModelPackage.get(model_package_name=MODEL_PKG_ARN)
print(f"status: {model_package.model_package_status}")

# To find the package later without the job object:
#   aws sagemaker list-model-packages --model-package-group-name seqkd-qwen35-4b-shopping \
#     --sort-by CreationTime --sort-order Descending \
#     --query 'ModelPackageSummaryList[0].ModelPackageArn'

In [ ]:
%store MODEL_PKG_ARN

## 04. Evaluate the student

Evaluate the fine-tuned student in two companion notebooks, in order — each takes the Model Package ARN from step 03:

1. **`2-tool-call-accuracy-gate.ipynb`** (run first) — a cheap, fast **gate**. A managed SageMaker **Custom Scorer** job measures per-step tool-call accuracy: given the teacher's prompt, does the student emit the right tool call? If the student doesn't beat the base here, fix training before running the complete evaluation. This saves the time and cost of a complete evaluation run.
2. **`3-evaluate-shopping-agent.ipynb`** — the full evaluation, which runs two complementary measures:
   - **Agent Success Rate (ASR)** — the headline metric. It runs the student through the full ShoppingBench agent loop on **held-out** problems and scores whether its final recommendation satisfies every constraint. This measures the capability we distilled.
   - **IFEval (regression check)** — general instruction-following, run to confirm that specializing the model for shopping tool-use didn't degrade its instruction-following ability.

Comparing the fine-tuned student against the base model on these axes tells you both how much task skill you gained and whether you preserved general competence.